In [1]:
import os
import torch
from dotenv import load_dotenv

load_dotenv()
access_token = os.environ.get("HF_TOKEN")
if access_token is None:
    raise ValueError("Set HF_TOKEN in your environment or in a local .env file.")

if torch.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

device

device(type='mps')

In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bigcode/starcoder2-3b")

test_config = {
    "tokenizer": tokenizer,
    "batch_size": 2,
    "context_length": 256,
    "vocab_size": len(tokenizer),
    "embedding_dim": 256,
    "n_layers": 4, 
    "n_heads": 4,
    "n_kv_heads": 2,
}

batch_size = test_config["batch_size"]
context_length = test_config["context_length"]
vocab_size = test_config["vocab_size"]
embedding_dim = test_config["embedding_dim"]
n_layers = test_config["n_layers"]
n_heads = test_config["n_heads"]
n_kv_heads = test_config["n_kv_heads"]

/Users/kuzeyaldemir/Projects/tiny-coder/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.


In [3]:
from data import create_python_dataset, CoderDataset
from torch.utils.data import DataLoader

small_python_dataset = create_python_dataset(access_token).take(200)
dataset = CoderDataset(small_python_dataset, tokenizer, context_length)

data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
data_loader_iterator = iter(data_loader)

In [4]:
x_sample, y_sample = next(data_loader_iterator)

In [5]:
x_sample, y_sample = x_sample.to(device), y_sample.to(device)

In [6]:
print(x_sample.shape)
print(y_sample.shape)

torch.Size([2, 256])
torch.Size([2, 256])


In [7]:
from model import LLM

model = LLM(vocab_size, context_length, embedding_dim, n_layers, n_heads, n_kv_heads, intermediate_dim=embedding_dim * 4).to(device)
logits = model(x_sample)
logits.shape

torch.Size([2, 256, 49152])

In [8]:
import torch.nn.functional as F

sample_loss = F.cross_entropy(torch.flatten(logits, end_dim=1), torch.flatten(y_sample))
sample_loss

tensor(173.7107, device='mps:0', grad_fn=<NllLossBackward0>)

In [9]:
import torch.optim as optim

optimizer = optim.AdamW(model.parameters(), lr=1e-3)

In [10]:
optimizer.zero_grad()
sample_loss.backward()
optimizer.step()

In [11]:
logits = model(x_sample)
new_loss = F.cross_entropy(torch.flatten(logits, end_dim=1), torch.flatten(y_sample))
new_loss

tensor(151.2993, device='mps:0', grad_fn=<NllLossBackward0>)

In [12]:
import torch.nn.functional as F

def train_one_epoch(data_loader, model, optimizer, max_steps=500):
    i = 0
    for x, y in iter(data_loader):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()

        logits = model(x)
        loss = F.cross_entropy(torch.flatten(logits, end_dim=1), torch.flatten(y))
        loss.backward()
        optimizer.step()
        i += 1
        if i % 10 == 0:
            print(loss)
        if i >= max_steps:
            break

train_one_epoch(data_loader, model, optimizer)

tensor(44.0001, device='mps:0', grad_fn=<NllLossBackward0>)
tensor(38.3469, device='mps:0', grad_fn=<NllLossBackward0>)
tensor(32.3562, device='mps:0', grad_fn=<NllLossBackward0>)
tensor(39.2176, device='mps:0', grad_fn=<NllLossBackward0>)
tensor(26.7988, device='mps:0', grad_fn=<NllLossBackward0>)
tensor(24.0741, device='mps:0', grad_fn=<NllLossBackward0>)
tensor(22.6163, device='mps:0', grad_fn=<NllLossBackward0>)
tensor(23.6513, device='mps:0', grad_fn=<NllLossBackward0>)
tensor(20.8850, device='mps:0', grad_fn=<NllLossBackward0>)
tensor(20.5836, device='mps:0', grad_fn=<NllLossBackward0>)
tensor(19.5145, device='mps:0', grad_fn=<NllLossBackward0>)
tensor(18.4074, device='mps:0', grad_fn=<NllLossBackward0>)
tensor(18.9147, device='mps:0', grad_fn=<NllLossBackward0>)
tensor(20.5039, device='mps:0', grad_fn=<NllLossBackward0>)
tensor(14.3092, device='mps:0', grad_fn=<NllLossBackward0>)
tensor(13.1580, device='mps:0', grad_fn=<NllLossBackward0>)
tensor(13.4065, device='mps:0', grad_fn=

In [13]:
eval_test_text = "def hello_world"
eval_input = tokenizer.encode(eval_test_text, return_tensors="pt").to(device)
eval_input.shape

torch.Size([1, 4])

In [14]:
eval_output = model(eval_input)
eval_output.shape

torch.Size([1, 4, 49152])

In [15]:
import torch

next_token_pred = torch.argmax(eval_output[:, -1, :], dim=-1).unsqueeze(1)
next_token_pred

tensor([[51]], device='mps:0')

In [16]:
new_sequence = torch.cat((eval_input, next_token_pred), dim=1)
new_sequence

tensor([[  610, 17966,   100,  5879,    51]], device='mps:0')

In [17]:
tokenizer.decode(new_sequence)

['def hello_world.']